# Train an image classifier

The training page runs the same TensorFlow this kernel has. The difference is
that here you can stop after epoch three, change the model, and run it again
without re-uploading anything — and every epoch still lands in the **Runs**
rail on the right, because `orinth.runs` writes where the platform reads.

In [ ]:
import orinth

trainable = [
    dataset
    for dataset in orinth.datasets.list()
    if dataset.task_type == "classification" and dataset.readiness.trainable
]
for dataset in trainable:
    print(f"{dataset.id:42} {dataset.total_items:>6} items  {dataset.splits}")

# The largest one, because a 70-item dataset trains to chance and teaches
# nothing about whether the pipeline works.
DATASET_ID = max(trainable, key=lambda dataset: dataset.total_items).id
ref = orinth.datasets.get(DATASET_ID)
LABELS = ref.labels
print("\ntraining on", ref.name, "|", LABELS)

## Read the images

`load()` gives one row per item with `path`, `label`, `width` and `height` —
the labels are already resolved, so nothing here has to know how the dataset was
laid out on disk. `limit=` keeps the arrays in memory; drop it once you know the
split fits, and reach for `orinth.datasets.records()` when it does not.

In [ ]:
import numpy as np
from PIL import Image

SIZE = (160, 160)
INDEX = {label: position for position, label in enumerate(LABELS)}


def arrays(split, limit):
    frame = orinth.datasets.load(DATASET_ID, split, limit=limit)
    images, targets = [], []
    for row in frame.iter_rows(named=True):
        if row["label"] not in INDEX:
            continue
        with Image.open(row["path"]) as handle:
            pixels = np.asarray(handle.convert("RGB").resize(SIZE), dtype="float32")
        images.append(pixels / 255.0)
        targets.append(INDEX[row["label"]])
    return np.stack(images), np.array(targets)


x_train, y_train = arrays("train", limit=600)
x_valid, y_valid = arrays("valid", limit=200)
print(x_train.shape, "train ·", x_valid.shape, "valid ·", len(LABELS), "classes")
print("class balance:", np.bincount(y_train, minlength=len(LABELS)))

## The model

Small on purpose. A three-block CNN trains in a minute on CPU and tells you
whether the *data* is learnable, which is the question worth answering before
anyone spends an afternoon on a bigger backbone. Swap in
`tf.keras.applications.MobileNetV3Small(include_top=False, weights="imagenet")`
when the answer is yes.

In [ ]:
import tensorflow as tf

model = tf.keras.Sequential(
    [
        tf.keras.layers.Input(shape=(*SIZE, 3)),
        tf.keras.layers.RandomFlip("horizontal"),
        tf.keras.layers.RandomRotation(0.05),
        tf.keras.layers.Conv2D(32, 3, activation="relu", padding="same"),
        tf.keras.layers.MaxPooling2D(),
        tf.keras.layers.Conv2D(64, 3, activation="relu", padding="same"),
        tf.keras.layers.MaxPooling2D(),
        tf.keras.layers.Conv2D(128, 3, activation="relu", padding="same"),
        tf.keras.layers.GlobalAveragePooling2D(),
        tf.keras.layers.Dropout(0.3),
        tf.keras.layers.Dense(len(LABELS), activation="softmax"),
    ]
)
model.compile(
    optimizer=tf.keras.optimizers.Adam(1e-3),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"],
)
model.summary()

## Train, logging as it goes

The callback is the whole integration: `run.log(step, **metrics)` per epoch, and
the chart in the rail fills in while `fit` is still running. Logging at the end
instead would give you one point and no way to see divergence.

In [ ]:
EPOCHS = 8


class LogToOrinth(tf.keras.callbacks.Callback):
    """One point per epoch, into the run this notebook owns."""

    def __init__(self, run):
        super().__init__()
        self.run = run

    def on_epoch_end(self, epoch, logs=None):
        self.run.log(epoch, **{key: float(value) for key, value in (logs or {}).items()})


with orinth.runs.start(
    "cnn baseline",
    params={"size": list(SIZE), "epochs": EPOCHS, "optimizer": "adam", "lr": 1e-3},
    dataset_id=DATASET_ID,
) as run:
    history = model.fit(
        x_train,
        y_train,
        validation_data=(x_valid, y_valid),
        epochs=EPOCHS,
        batch_size=32,
        callbacks=[LogToOrinth(run)],
        verbose=2,
    )
    run.log_text("labels", ", ".join(LABELS))

    weights = run.directory / "model.keras"
    model.save(weights)
    run.log_artifact(weights, name="model.keras")
    print("saved", weights.name, "to run", run.id)

## Where the curve went

Train accuracy far above validation is overfitting, and at this size the fix is
more data or more augmentation before it is a bigger model. The run is on disk
either way — open it from the rail and the numbers are still there tomorrow.

In [ ]:
%matplotlib inline
import matplotlib.pyplot as plt

figure, axes = plt.subplots(1, 2, figsize=(9, 3.2))
for axis, key in zip(axes, ["loss", "accuracy"]):
    axis.plot(history.history[key], label=f"train {key}")
    axis.plot(history.history[f"val_{key}"], label=f"valid {key}")
    axis.set_xlabel("epoch")
    axis.legend()
figure.tight_layout()
plt.show()